# 🫁 Pneumonia Detection - Master Execution Notebook

Welcome to the master execution notebook! This notebook is designed to run the entire Pneumonia Detection pipeline on **either Google Colab (GPU accelerated)** or **your local machine (CPU/GPU)**. The environment is automatically detected, and paths/instructions adjust accordingly.

### ⚡ Architecture Overview
- **Google Colab (GPU Runtime)**: Highly recommended for training models (DenseNet121+Swin Ensemble CheX-DS takes ~38 mins on GPU vs ~9.3 hours on CPU). Mounts Google Drive to persist weights and saves the dataset to the local scratch disk (`/content/data/`) to bypass Drive latency.
- **Local Machine (CPU/GPU)**: Best for evaluation, inference, and running the Streamlit app. Re-uses the weights generated on Colab.

---

### 🚀 Google Colab Setup:
1. **Enable GPU**: Click on **Runtime** -> **Change runtime type** -> select **GPU** (T4, L4, or A100) -> click **Save**.
2. **Upload Folder**: Upload the `pneumonia-detection-cnn-xray` folder directly under your Google Drive's **My Drive/** folder. 
   *   ⚠️ **CRITICAL: Exclude the `data/` and `venv/` directories when uploading!** Excluding these directories keeps the upload size tiny and takes only a few seconds.

## Step 1: Environment Detection & Google Drive Mount
Detects if the code is running on Google Colab or a local machine. If running on Colab, it mounts Google Drive.

In [ ]:
import sys
import os

# Detect environment
try:
    import google.colab
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if IS_COLAB:
    print("📍 Google Colab environment detected. Mounting Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print("🖥️ Local machine detected. Skipping Google Drive mount.")

## Step 2: Set Workspace Directory
Sets the current working directory to the project root directory.

In [ ]:
if IS_COLAB:
    project_dir = '/content/drive/MyDrive/pneumonia-detection-cnn-xray'
    if not os.path.exists(project_dir):
        print(f"❌ ERROR: Project folder '{project_dir}' was not found in your Google Drive.")
        print("Please verify that you uploaded 'pneumonia-detection-cnn-xray' directly under 'My Drive'.")
    else:
        %cd {project_dir}
        print("\n📁 Google Drive Workspace Directory Contents:")
        !ls -la
else:
    print("🖥️ Running locally. Current working directory:")
    %pwd
    print("\n📁 Local Directory Contents:")
    print(os.listdir('.'))

## Step 3: Hardware Verification & Package Installation
Verifies hardware runtime support (GPU vs CPU) and installs required Python libraries.

In [ ]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"GPU (CUDA) Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")
else:
    if IS_COLAB:
        print("⚠️ WARNING: GPU not detected. Training will be extremely slow. Enable GPU in Runtime -> Change runtime type.")
    else:
        print("ℹ️ Running on CPU (standard for local development/inference).")

In [ ]:
# Install project dependencies
import sys, subprocess

def pip_install(pkgs): subprocess.check_call([sys.executable, "-m", "pip", "install"] + pkgs)

try:
    import google.colab
    print("📍 Google Colab detected.\nInstalling auxiliary packages (Streamlit, GitPython)...\n")
    pip_install(["streamlit", "GitPython"])
    print("✅ Done (no runtime restart required).")
except ImportError:
    print("🖥️ Local environment detected.\nInstalling dependencies from requirements.txt...\n")
    pip_install(["-r", "requirements.txt"])
    print("✅ Done.")

### Step 3b: Verify Model Architecture Compilation
Verifies that all model architectures build correctly on the active device.

In [ ]:
!python test_model.py

## Step 4: High-Performance Data Extraction
Downloads the dataset (if missing) and extracts it. Checks file size integrity using S3 HTTP HEAD values to ensure incomplete downloads are overwritten.

In [ ]:
# Run the download and extraction script
!python download_data.py

In [ ]:
# Verify that data files were downloaded locally
import os
from src.config import DATA_DIR

print(f"Checking dataset structure at: {DATA_DIR}")
if os.path.exists(DATA_DIR):
    print("Dataset folder structure:", os.listdir(DATA_DIR))
    train_path = os.path.join(DATA_DIR, 'train')
    if os.path.exists(train_path):
        print("Train subfolders:", os.listdir(train_path))
else:
    print("❌ Dataset folder not found. Please run download_data.py above.")

### Step 4b: Verify Data Loading & Dataset Pipeline
Runs a pre-flight test verifying data loaders can fetch a batch of scans.

In [ ]:
!python test_data.py

## Step 5: Train Models
Runs the training pipelines. Weight files (`pneumonia_model_{model}.pth`) are saved directly to `models/`.

### Option A: Train a Single Model
Run one of the cells below to train a specific model architecture.

In [ ]:
# Train baseline SimpleCNN model
!python main.py --model cnn

In [ ]:
# Train Transfer Learning (ResNet50) model
!python main.py --model resnet

In [ ]:
# Train CheX-DS (DenseNet + Swin Transformer Ensemble - Recommended)
!python main.py --model chexds

### Option B: Run Full Benchmarks (All Models)
Trains all models sequentially on the GPU, logging the results in `results/experiments_summary.json`.

In [ ]:
!python run_experiments.py

## Step 6: Evaluation & Result Visualization
Runs evaluation and generates classification reports and Confusion Matrix/ROC curve plots in `results/`.

In [ ]:
%matplotlib inline
# Choose model to evaluate: cnn, resnet, or chexds
!python visualize_results.py --model chexds

## Step 7: Run Streamlit Web Application inside Colab (Optional)
Exposes the Streamlit interface using `localtunnel` port forwarding. **Skip this step if running locally.**

In [ ]:
if IS_COLAB:
    # 1. Install localtunnel
    !npm install -g localtunnel
else:
    print("🖥️ Running locally. Skip localtunnel setup and run Step 8 directly.")

In [ ]:
if IS_COLAB:
    # 2. Find your Colab public IP address (used as the password for localtunnel verification)
    import urllib
    print("Your Colab Endpoint Password:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip())
else:
    print("🖥️ Running locally.")

In [ ]:
if IS_COLAB:
    # 3. Run Streamlit with localtunnel
    # Click the URL generated below, enter the 'Endpoint Password', and click submit.
    !streamlit run app.py & npx localtunnel --port 8501
else:
    print("🖥️ Running locally. Use Step 8 command to start dashboard.")

## Step 8: Run Streamlit App Locally (Recommended)
If you trained your models on Google Colab, download the generated weight files (`pneumonia_model_*.pth`) from your Google Drive into your local `models/` directory.

To start the Streamlit web dashboard locally on your PC:
1. Open a terminal in your project root directory.
2. Run the application:
   ```bash
   streamlit run app.py
   ```
3. Open `http://localhost:8501` in your web browser.
4. Choose between `CheX-DS`, `ResNet50`, and `Simple CNN` architectures dynamically in the sidebar dropdown!